In [1]:
from datasets import load_dataset
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize
import random
import pandas as pd
import datetime

/Users/ninahew/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/ninahew/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
import numpy as np
desired_width=320
pd.set_option('display.width', desired_width)
np.set_printoptions(linewidth=desired_width)
pd.set_option('display.max_columns',10)

In [3]:
def generate_dict(x, lang):
    lang_ds = load_dataset("wikimedia/wikipedia", "20231101.{lang}".format(lang=lang))
    print('dataset ' + '{lang}'.format(lang=lang) + ' loaded')
    # Consider returning number of rows in datasets
    corpus_lang = lang_ds['train']['text'][0:x]
    # print(corpus_lang)
    # Convert to big string
    string_lang = " ".join(corpus_lang)
    tokenized_lang= word_tokenize(string_lang)
    print("{lang}".format(lang=lang) + " after tokenisation")
    print("{lang}".format(lang=lang), " : ", str(len(tokenized_lang)), " words")
    # Remove punctuation, proper nouns and numerical entries
    cleaned_lang = [word for word in tokenized_lang if ((len(word) > 1) and (not(word[0].isupper())) and (not(word.isnumeric())))]
    print("{lang}".format(lang=lang) + " after data cleaning")
    print("{lang}".format(lang=lang), " : ", str(len(cleaned_lang)), " words")
    dict_lang = {}
    for word in cleaned_lang:
        if word not in dict_lang:
            dict_lang[word] = 0
        dict_lang[word] += 1
    return dict_lang


In [4]:
vls = generate_dict(100, 'vls')
nl = generate_dict(100, 'nl')


dataset vls loaded
vls after tokenisation
vls  :  37159  words
vls after data cleaning
vls  :  22845  words
dataset nl loaded
nl after tokenisation
nl  :  213125  words
nl after data cleaning
nl  :  149607  words


In [5]:
corpus_vls_nl = {'west flemish' : vls, 'nederlands' : nl}

In [6]:
# Creating test function using small sample data, before using wikipedia data

# corpus_1 = {'en' : {'cat' : 2, 'dog' : 5, 'lizard' : 7},
#           'fr' : {'chat' : 4, 'chien' : 2, 'lizard' : 10}}

# Function takes a word and a corpus and returns its language
def identify_language(given_word, corpus):
    current_language = 'unknown'
    highest_frequency = 0
    for language_name, word_frequencies in corpus.items():
        if given_word in word_frequencies:
            if word_frequencies[given_word] > highest_frequency:
                current_language = language_name
                highest_frequency = word_frequencies[given_word]

    return current_language, highest_frequency




In [7]:
def generate_test_phrase(x: int, lang: str, phrase_length: int):
    lang_ds = load_dataset("wikimedia/wikipedia", "20231101.{lang}".format(lang=lang))
    # print('dataset loaded')
    corpus_lang = lang_ds['train']['text'][0:x]
    # Convert to big string
    string_lang = " ".join(corpus_lang)
    # print(str(len(string_lang)) + ' words in string_lang')
    tokenized_lang= word_tokenize(string_lang)
    # print(str(len(tokenized_lang)) + ' words in token lang')
    cleaned_lang = [word for word in tokenized_lang if (len(word) > 1)]
    # print(str(len(cleaned_lang)) + 'words in clean lang')
    no_proper_nouns_lang = [word for word in cleaned_lang if not word[0].isupper()]
    no_numbers_lang = [word for word in no_proper_nouns_lang if not word.isnumeric()]
    random_number = random.randint(0,(len(no_numbers_lang) - phrase_length))
    return no_numbers_lang[random_number : random_number+phrase_length]

In [8]:
w_flemish_test_phrase_1 = generate_test_phrase(100, 'vls', 10)
print(w_flemish_test_phrase_1)

['kan', 'worren', 'kiek', 'mo', 'no', "z'n", 'palmarès', 'issen', 'carrière', 'spietig']


In [9]:
[identify_language(word, corpus_vls_nl) for word in w_flemish_test_phrase_1]

[('nederlands', 237),
 ('west flemish', 1),
 ('west flemish', 1),
 ('west flemish', 27),
 ('west flemish', 25),
 ('west flemish', 9),
 ('west flemish', 1),
 ('west flemish', 1),
 ('nederlands', 10),
 ('west flemish', 1)]

In [10]:
nederlands_test_phrase_1 = generate_test_phrase(100, 'nl', 10)
print(nederlands_test_phrase_1)

['het', 'organiseren', 'van', 'de', 'voor', 'gehandicapten', 'werd', 'in', 'de', 'gastheer']


In [11]:
[identify_language(word, corpus_vls_nl) for word in nederlands_test_phrase_1]

[('nederlands', 4810),
 ('nederlands', 9),
 ('nederlands', 7101),
 ('nederlands', 12367),
 ('nederlands', 1309),
 ('nederlands', 4),
 ('nederlands', 971),
 ('nederlands', 4474),
 ('nederlands', 12367),
 ('nederlands', 6)]

In [12]:
result = [identify_language(word, corpus_vls_nl) for word in nederlands_test_phrase_1]
# print(result)


# print(result[0][0])
language_occurrence = [lang for lang, freq in result]
s = pd.Series(language_occurrence)
# print(s.mode().to_string().strip())


In [13]:
def sentence_language_identifier(sentence: list[str], corpus):
    list_languages = [identify_language(word, corpus) for word in sentence]
    language_occurrence = [lang for lang, freq in list_languages]
    # print(language_occurrence)
    lang_occ_series = pd.Series(language_occurrence)
    # print(lang_occ_series)
    modal_lang = lang_occ_series.mode()
    if modal_lang.empty:
        return 'unknown'
    elif 'nederland' in modal_lang.to_string():
        return 'nl'
    else:
        return 'vls'

In [14]:
nl1 = generate_test_phrase(10, 'nl', 10)
sentence_language_identifier(nl1, corpus_vls_nl)

'nl'

In [15]:
vls = generate_test_phrase(10, 'vls', 10)
sentence_language_identifier(vls, corpus_vls_nl)

'vls'

In [17]:
# Testing dutch sentences

results = {}

for fold in range(25,150,25):
    for i in range(1,11):
        test_sentence = generate_test_phrase(fold ,'nl', i)
        start_time = datetime.datetime.now()
        predicted_language = sentence_language_identifier(test_sentence, corpus_vls_nl)
        end_time = datetime.datetime.now()
        results['Fold'] = fold
        results['Sentence_length'] = len(test_sentence)
        results['Language'] = predicted_language
        results['Time taken'] = end_time - start_time
      # print('Corpus made of ', article_number, ' articles')
        # print('Sentence made of ', sentence_number, ' words')
        # print('Language is', result.upper())
        # print('time taken: ', time_taken)
        print(results)



{'Fold': 25, 'Sentence_length': 1, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=538)}
{'Fold': 25, 'Sentence_length': 2, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=535)}
{'Fold': 25, 'Sentence_length': 3, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=544)}
{'Fold': 25, 'Sentence_length': 4, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=521)}
{'Fold': 25, 'Sentence_length': 5, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=621)}
{'Fold': 25, 'Sentence_length': 6, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=525)}
{'Fold': 25, 'Sentence_length': 7, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=529)}
{'Fold': 25, 'Sentence_length': 8, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=539)}
{'Fold': 25, 'Sentence_length': 9, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=525)}
{'Fold': 25, 'Sentence_length': 10, 'Language': 'nl', '

In [18]:
# Testing Flemish sentences

results = {}

for fold in range(25,150,25):
    for i in range(1,11):
        test_sentence = generate_test_phrase(fold ,'vls', i)
        start_time = datetime.datetime.now()
        predicted_language = sentence_language_identifier(test_sentence, corpus_vls_nl)
        end_time = datetime.datetime.now()
        results['Fold'] = fold
        results['Sentence_length'] = len(test_sentence)
        results['Language'] = predicted_language
        results['Time taken'] = end_time - start_time
      # print('Corpus made of ', article_number, ' articles')
        # print('Sentence made of ', sentence_number, ' words')
        # print('Language is', result.upper())
        # print('time taken: ', time_taken)
        print(results)



{'Fold': 25, 'Sentence_length': 1, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=504)}
{'Fold': 25, 'Sentence_length': 2, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=568)}
{'Fold': 25, 'Sentence_length': 3, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=512)}
{'Fold': 25, 'Sentence_length': 4, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=534)}
{'Fold': 25, 'Sentence_length': 5, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=529)}
{'Fold': 25, 'Sentence_length': 6, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=522)}
{'Fold': 25, 'Sentence_length': 7, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=512)}
{'Fold': 25, 'Sentence_length': 8, 'Language': 'nl', 'Time taken': datetime.timedelta(microseconds=637)}
{'Fold': 25, 'Sentence_length': 9, 'Language': 'vls', 'Time taken': datetime.timedelta(microseconds=549)}
{'Fold': 25, 'Sentence_length': 10, 'Language': 'v

In [19]:
from nltk.classify.textcat import TextCat

In [20]:
# refactoring existing LID tool to include TextCat results in the output, testing both models on the same data
# Benchmarking Dutch sentences
tc = TextCat()

results_list = []

for fold in range(25,150,25):
    for i in range(1,11):
        result_dict = {}
        test_sentence = generate_test_phrase(fold ,'nl', i)
        start_time_rule_based = datetime.datetime.now()
        predicted_language_rule_based = sentence_language_identifier(test_sentence, corpus_vls_nl)
        end_time_rule_based = datetime.datetime.now()
        result_dict['Fold'] = fold
        result_dict['Sentence_length'] = len(test_sentence)
        result_dict['Language (RB)'] = predicted_language_rule_based
        result_dict['Time taken (RB)'] = end_time_rule_based - start_time_rule_based
        test_sentence_joined = " ".join(test_sentence)
        # print(test_sentence_joined)
        start_time_textcat = datetime.datetime.now()
        predicted_language_textcat = tc.guess_language(test_sentence_joined)
        end_time_textcat = datetime.datetime.now()
        result_dict['Language (TC)'] = predicted_language_textcat
        result_dict['Time taken (TC)'] = end_time_textcat - start_time_textcat
        results_list.append(result_dict)

df = pd.DataFrame(results_list)
print(df)



    Fold  Sentence_length Language (RB)        Time taken (RB) Language (TC)        Time taken (TC)
0     25                1            nl 0 days 00:00:00.000738           ojw 0 days 00:00:00.047243
1     25                2            nl 0 days 00:00:00.000543           ast 0 days 00:00:00.070739
2     25                3            nl 0 days 00:00:00.000624           afr 0 days 00:00:00.131546
3     25                4            nl 0 days 00:00:00.000526           vls 0 days 00:00:00.085130
4     25                5            nl 0 days 00:00:00.000533           afr 0 days 00:00:00.177559
5     25                6            nl 0 days 00:00:00.000547           vls 0 days 00:00:00.137467
6     25                7            nl 0 days 00:00:00.000715           vls 0 days 00:00:00.243236
7     25                8            nl 0 days 00:00:00.000561           zea 0 days 00:00:00.124104
8     25                9            nl 0 days 00:00:00.000520           sot 0 days 00:00:00.306928


In [21]:
# Dutch
print(df.groupby(['Language (RB)'])['Fold'].count())

Language (RB)
nl    50
Name: Fold, dtype: int64


In [22]:
# Dutch
print(df.groupby(['Language (TC)'])['Fold'].count())

Language (TC)
afr    22
ast     1
cym     1
fri     1
nds     1
nld     3
ojw     1
sot     1
sun     2
swe     1
vls    14
zea     2
Name: Fold, dtype: int64


In [23]:
# Benchmarking West Flemish sentences
tc = TextCat()

results_list = []

for fold in range(25,150,25):
    for i in range(1,11):
        result_dict = {}
        test_sentence = generate_test_phrase(fold ,'vls', i)
        start_time_rule_based = datetime.datetime.now()
        predicted_language_rule_based = sentence_language_identifier(test_sentence, corpus_vls_nl)
        end_time_rule_based = datetime.datetime.now()
        result_dict['Fold'] = fold
        result_dict['Sentence_length'] = len(test_sentence)
        result_dict['Language (RB)'] = predicted_language_rule_based
        result_dict['Time taken (RB)'] = end_time_rule_based - start_time_rule_based
        test_sentence_joined = " ".join(test_sentence)
        # print(test_sentence_joined)
        start_time_textcat = datetime.datetime.now()
        predicted_language_textcat = tc.guess_language(test_sentence_joined)
        end_time_textcat = datetime.datetime.now()
        result_dict['Language (TC)'] = predicted_language_textcat
        result_dict['Time taken (TC)'] = end_time_textcat - start_time_textcat
        results_list.append(result_dict)


df = pd.DataFrame(results_list)
print(df)

    Fold  Sentence_length Language (RB)        Time taken (RB) Language (TC)        Time taken (TC)
0     25                1            nl 0 days 00:00:00.000506           gym 0 days 00:00:00.031468
1     25                2           vls 0 days 00:00:00.000792           fri 0 days 00:00:00.073884
2     25                3           vls 0 days 00:00:00.000515           vls 0 days 00:00:00.117236
3     25                4            nl 0 days 00:00:00.000559           nds 0 days 00:00:00.081759
4     25                5           vls 0 days 00:00:00.000504           vls 0 days 00:00:00.126151
5     25                6           vls 0 days 00:00:00.000511           vls 0 days 00:00:00.139018
6     25                7           vls 0 days 00:00:00.000513           vls 0 days 00:00:00.182202
7     25                8           vls 0 days 00:00:00.000514           vls 0 days 00:00:00.158462
8     25                9           vls 0 days 00:00:00.000541           vls 0 days 00:00:00.351462


In [24]:
# West Flemish
print(df.groupby(['Language (RB)'])['Fold'].count())

Language (RB)
nl     23
vls    27
Name: Fold, dtype: int64


In [25]:
# West Flemish
print(df.groupby(['Language (TC)'])['Fold'].count())

Language (TC)
abk     1
afr     1
ame     1
deu     1
eng     3
fra     2
fri     2
gym     1
kos     1
nds     3
nld     2
nob     1
sco     1
spa     1
sun     1
vls    28
Name: Fold, dtype: int64
